# 13 — Information Retrieval & BM25
**Goal:** Construir un motor de búsqueda de texto que rankee documentos mejor que TF-IDF coseno.

## BM25 (Best Match 25)

El estándar industrial para búsqueda desde los años 90. Mejora TF-IDF en dos aspectos:

$$\text{BM25}(d, q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{\text{TF}(t,d) \cdot (k_1+1)}{\text{TF}(t,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

- **TF saturation**: $k_1$ controla cuánto vale repetir una palabra (default 1.5)
- **Length normalization**: $b$ penaliza documentos largos (default 0.75)
- Sin normalizar: TF crece sin límite; con BM25: se satura

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

STOPWORDS_ES = [
    'a','al','como','con','de','del','el','en','es','la','las','lo','los',
    'me','mi','muy','no','o','para','por','que','se','si','sin','su','te',
    'todo','un','una','y','yo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

## 1. BM25 desde cero

In [ ]:
class BM25:
    """BM25 Okapi — Robertson et al. (1994)."""

    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b  = b

    def fit(self, corpus, preprocessor=None):
        self.preprocessor = preprocessor or (lambda x: x.lower())
        self.docs  = [self.preprocessor(d).split() for d in corpus]
        self.N     = len(self.docs)
        self.avgdl = np.mean([len(d) for d in self.docs])

        # Document frequency de cada término
        self.df = Counter()
        for doc in self.docs:
            self.df.update(set(doc))

        # IDF con suavizado BM25
        # IDF(t) = log((N - df + 0.5) / (df + 0.5) + 1)
        self.idf = {
            t: np.log((self.N - c + 0.5) / (c + 0.5) + 1)
            for t, c in self.df.items()
        }
        return self

    def score(self, query, doc_idx):
        doc    = self.docs[doc_idx]
        dl     = len(doc)
        tf_map = Counter(doc)
        score  = 0.0
        for term in query.split():
            if term not in self.idf:
                continue
            tf    = tf_map.get(term, 0)
            # TF saturada
            tf_sat = tf * (self.k1 + 1) / (
                tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            )
            score += self.idf[term] * tf_sat
        return score

    def search(self, query, top_k=5):
        query = self.preprocessor(query)
        scores = np.array([self.score(query, i) for i in range(self.N)])
        top    = np.argsort(scores)[::-1][:top_k]
        return list(zip(top, scores[top]))

print('BM25 implementado.')

In [ ]:
# Corpus de tickets de soporte
corpus = [
    'La app está muy lenta y tarda en cargar la pantalla',
    'La aplicación carga lento, no puedo entrar al sistema',
    'La app es rápida y fácil de usar',
    'No me llegó el OTP al celular para confirmar',
    'El código OTP no llega por SMS, intenté varias veces',
    'El OTP llegó al instante, todo funcionó bien',
    'La carga de documentos falla constantemente en la app',
    'No puedo subir los documentos requeridos al sistema',
    'Subir mis documentos fue sencillo y rápido',
    'Aprobaron mi crédito en minutos, proceso muy ágil',
    'Mi solicitud de crédito lleva días en evaluación',
    'La evaluación crediticia no tiene respuesta después de semanas',
    'El soporte al cliente no responde mis mensajes',
    'La atención al cliente fue excelente y muy rápida',
    'Nadie del soporte resuelve mi problema después de tres días',
    'Excelente servicio, recomiendo la tarjeta a todos',
    'La tarjeta tiene increíbles beneficios y descuentos',
    'Pésima experiencia con la tarjeta, no recomiendo para nada',
]

bm25 = BM25(k1=1.5, b=0.75).fit(corpus, preprocessor=preprocess)

# TF-IDF coseno para comparar
tfidf = TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES)
X = tfidf.fit_transform(corpus)

def search_tfidf(query, top_k=5):
    q = tfidf.transform([query])
    sims = cosine_similarity(q, X).ravel()
    top  = np.argsort(sims)[::-1][:top_k]
    return list(zip(top, sims[top]))

# Comparar en una query
query = 'la app no funciona está lenta'

print(f'Query: "{query}"\n')
print('BM25:')
for idx, score in bm25.search(query, top_k=4):
    print(f'  [{score:.3f}] {corpus[idx]}')

print('\nTF-IDF coseno:')
for idx, score in search_tfidf(query, top_k=4):
    print(f'  [{score:.3f}] {corpus[idx]}')

## 2. TF saturation — por qué BM25 es mejor

In [ ]:
# TF-IDF: TF crece linealmente con las repeticiones
# BM25: TF se satura → la palabra 10ª repetición aporta mucho menos

tf_values = np.arange(0, 20, 0.1)

# TF puro
tf_pure = tf_values

# BM25 TF con distintos k1
def bm25_tf(tf, k1, b=0, dl=1, avgdl=1):
    return tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / avgdl))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(tf_values, tf_pure, label='TF puro (TF-IDF)', color='#aaa', linewidth=2, linestyle='--')
for k1, color in [(0.5, '#4361ee'), (1.2, '#06d6a0'), (2.0, '#f72585')]:
    ax.plot(tf_values, bm25_tf(tf_values, k1), label=f'BM25 k₁={k1}', linewidth=2, color=color)

ax.set_xlabel('Frecuencia del término (TF)')
ax.set_ylabel('Contribución al score')
ax.set_title('TF saturation — BM25 vs TF puro')
ax.legend()
ax.set_xlim(0, 20)
plt.tight_layout()
plt.show()

print('Con BM25, repetir una palabra 20 veces aporta poco más que 5 veces.')
print('Con TF puro, 20 repeticiones valen exactamente 4× más que 5.')

## 3. Evaluación de ranking — métricas IR

In [ ]:
# Precision@k, Recall@k, MRR, NDCG

def precision_at_k(ranked_docs, relevant_docs, k):
    top_k = set(ranked_docs[:k])
    return len(top_k & set(relevant_docs)) / k

def recall_at_k(ranked_docs, relevant_docs, k):
    top_k = set(ranked_docs[:k])
    return len(top_k & set(relevant_docs)) / len(relevant_docs)

def reciprocal_rank(ranked_docs, relevant_docs):
    for i, doc in enumerate(ranked_docs):
        if doc in relevant_docs:
            return 1 / (i + 1)
    return 0.0

def dcg_at_k(ranked_docs, relevances, k):
    """Discounted Cumulative Gain."""
    dcg = 0.0
    for i, doc in enumerate(ranked_docs[:k]):
        rel = relevances.get(doc, 0)
        dcg += rel / np.log2(i + 2)   # log(rank + 1)
    return dcg

def ndcg_at_k(ranked_docs, relevances, k):
    """Normalized DCG — divide por el DCG ideal."""
    actual_dcg = dcg_at_k(ranked_docs, relevances, k)
    ideal_order = sorted(relevances, key=relevances.get, reverse=True)
    ideal_dcg   = dcg_at_k(ideal_order, relevances, k)
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

# Evaluar con queries y relevances anotadas manualmente
queries_eval = [
    {
        'query':    'app lenta no carga',
        'relevant': {0, 1},    # índices de docs relevantes
        'grades':   {0: 3, 1: 2, 2: 1},  # relevancia graduada (0-3)
    },
    {
        'query':    'OTP no llega SMS',
        'relevant': {3, 4},
        'grades':   {3: 3, 4: 3, 5: 1},
    },
    {
        'query':    'soporte cliente no responde',
        'relevant': {12, 14},
        'grades':   {12: 3, 14: 3, 13: 1},
    },
]

k = 5
print(f'Evaluación — k={k}\n')
print(f'{"":<20} {"P@k":>6} {"R@k":>6} {"MRR":>6} {"NDCG@k":>8}')
print('-' * 55)

for system_name, search_fn in [('BM25', lambda q: [i for i, _ in bm25.search(q, top_k=k)]),
                                ('TF-IDF', lambda q: [i for i, _ in search_tfidf(q, top_k=k)])]:
    ps, rs, rrs, ndcgs = [], [], [], []
    for qdict in queries_eval:
        ranked = search_fn(qdict['query'])
        rel    = qdict['relevant']
        grades = qdict['grades']
        ps.append(precision_at_k(ranked, rel, k))
        rs.append(recall_at_k(ranked, rel, k))
        rrs.append(reciprocal_rank(ranked, rel))
        ndcgs.append(ndcg_at_k(ranked, grades, k))
    print(f'{system_name:<20} {np.mean(ps):6.3f} {np.mean(rs):6.3f} {np.mean(rrs):6.3f} {np.mean(ndcgs):8.3f}')

## 4. Hiperparámetros BM25 — efecto de k₁ y b

In [ ]:
results_grid = []
for k1 in [0.5, 1.0, 1.5, 2.0, 3.0]:
    for b in [0.0, 0.25, 0.5, 0.75, 1.0]:
        bm = BM25(k1=k1, b=b).fit(corpus, preprocessor=preprocess)
        ndcgs = []
        for qdict in queries_eval:
            ranked = [i for i, _ in bm.search(qdict['query'], top_k=5)]
            ndcgs.append(ndcg_at_k(ranked, qdict['grades'], 5))
        results_grid.append({'k1': k1, 'b': b, 'ndcg': np.mean(ndcgs)})

df_grid = pd.DataFrame(results_grid)
pivot   = df_grid.pivot(index='k1', columns='b', values='ndcg')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, label='NDCG@5')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
ax.set_xlabel('b (length normalization)'); ax.set_ylabel('k₁ (TF saturation)')
ax.set_title('BM25 grid search — NDCG@5')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if pivot.values[i,j] > 0.7 else 'black')
plt.tight_layout()
plt.show()

best = df_grid.loc[df_grid.ndcg.idxmax()]
print(f'Mejores parámetros: k1={best.k1}, b={best.b}, NDCG={best.ndcg:.3f}')

## Resumen
| Concepto | Fórmula / API |
|---|---|
| BM25 score | $\sum_t \text{IDF}(t) \cdot \frac{\text{TF}(t,d)(k_1+1)}{\text{TF}(t,d)+k_1(1-b+b\frac{|d|}{\text{avgdl}})}$ |
| k₁ | Controla saturación de TF (default 1.5) |
| b | Penalización por longitud (0=ninguna, 1=total; default 0.75) |
| Precision@k | Fracción de relevantes en top-k |
| Recall@k | Fracción de todos los relevantes en top-k |
| MRR | Mean Reciprocal Rank — posición del primer relevante |
| NDCG@k | Considera relevancia graduada y posición del ranking |

**Siguiente:** `14_feature_engineering.ipynb` — features más allá de las palabras.